# Lee2019 data preparation and imputation tutorial

These examples use **Lee2019 contacts and hg38 genome resources** to prepare cell-number subsets, aggregate raw pseudo-bulk maps and generate single-cell or pseudo-bulk imputed inputs.

`BENCHMARK_ROOT` denotes the workspace prefix used in the command examples.


<a id="Inputs-and-cell-selection"></a>

## 1. Inputs and cell selection


<a id="1.-Read-metadata-and-contact-tables"></a>

### 1.1. Read metadata and contact tables

Metadata retains the original cell IDs and cell-type labels. The prepared
42-cell Embedding metadata contains the following records:

Source: `metadata.tsv` (first 3 lines).

```text
cell_id	cell_type	subtype
181218_snm3Cseq_hs_21yr_BA10_UMB5577_1_UMB5577_2_A10_AD001	L2/3	L2/3-1
181218_snm3Cseq_hs_21yr_BA10_UMB5577_1_UMB5577_2_A10_AD004	L2/3	L2/3-2
```

The ODC raw-contact table records one cell ID and one contact-file path per
line. The full table contains 1,245 cells. Its recorded server paths are
retained below:

Source: `cell_table.tsv` (first 2 lines).

```text
181218_snm3Cseq_hs_21yr_BA10_UMB5577_1_UMB5577_2_A10_AD007	/home/ygc/scHiCluster_Lee2019_hg38/raw/181218_snm3Cseq_hs_21yr_BA10_UMB5577_1_UMB5577_2_A10_AD007.3C.contact.rmbkl.tsv.gz
181218_snm3Cseq_hs_21yr_BA10_UMB5577_1_UMB5577_2_A12_AD001	/home/ygc/scHiCluster_Lee2019_hg38/raw/181218_snm3Cseq_hs_21yr_BA10_UMB5577_1_UMB5577_2_A12_AD001.3C.contact.rmbkl.tsv.gz
```

The 100-cell input has its own saved list:

Source: `cell_table_100.txt` (first 2 lines).

```text
190315_snm3Cseq_hs_21yr_BA10_UMB5577_3_UMB5577_4_A6_AD012	/home/inspur/hcx418000230017/reference/data/lee_2019/rmbkl/190315_snm3Cseq_hs_21yr_BA10_UMB5577_3_UMB5577_4_A6_AD012.contact.rmbkl.tsv.gz
190315_snm3Cseq_hs_21yr_BA10_UMB5577_5_UMB5577_6_G6_AD004	/home/inspur/hcx418000230017/reference/data/lee_2019/rmbkl/190315_snm3Cseq_hs_21yr_BA10_UMB5577_5_UMB5577_6_G6_AD004.contact.rmbkl.tsv.gz
```

The Embedding input uses five-column contacts at **100 kb**: chromosome 1,
bin-start position 1, chromosome 2, bin-start position 2 and contact count.
The first three lines of one input file are:

Source: `181218_snm3Cseq_hs_21yr_BA10_UMB5577_1_UMB5577_2_A10_AD001.txt` (first 3 lines).

```text
chr1	800000	chr1	800000	3
chr1	800000	chr1	900000	1
chr1	800000	chr1	1900000	1
```

The genome-size file contains chromosome names and lengths:

Source: `hg38.autosomal.chrom.sizes` (first 3 lines).

```text
chr1	248956422
chr2	242193529
chr3	198295559
```

Contact-column arguments must follow the input format. The five-column
100-kb table above retains aggregated counts; the raw-contact workflow uses
the column settings configured in its implementation.


<a id="2.-Filter-blacklisted-contacts"></a>

### 1.2. Filter blacklisted contacts

The single-cell imputation implementation ([data_preparation/single_cell_imputation/impute.py](https://github.com/rawandy2001-design/benchmark/blob/main/data_preparation/single_cell_imputation/impute.py) in this folder)
uses the same contact layout for filtering and `prepare-impute`:

```bash
hicluster filter-contact \
  --contact_table "$UNFILTERED_CONTACT_TABLE" \
  --chrom_size_path "${BENCHMARK_ROOT}/code/resources/genome/hg38.autosomal.chrom.sizes" \
  --blacklist_1d_path "$HG38_BLACKLIST" \
  --output_dir "$FILTERED_CONTACT_DIR" \
  --chr1 1 --pos1 2 --chr2 5 --pos2 6
```

Its `build_filtered_contact_table` block writes a new, headerless two-column
`contact_table_rmbkl.tsv`. Each ID points to the generated
`<cell_id>.contact.rmbkl.tsv.gz`, rather than its unfiltered source.
Keep that filtered table with the metadata and the original table.


<a id="3.-Select-cell-types-and-construct-nested-subsets"></a>

### 1.3. Select cell types and construct nested subsets

The downsampling implementation ([data_preparation/downsampling/downsampling.py](https://github.com/rawandy2001-design/benchmark/blob/main/data_preparation/downsampling/downsampling.py) in this folder) first joins
metadata to contact paths by cell ID. It selects up to the two largest cell
types with at least 100 cells. Ties are ordered by cell-type name.

For each selected type, the code sorts IDs, generates one permutation and
uses successive prefixes. The settings and core selection are:

```python
import numpy as np

MIN_CELLS = 100
CELL_STEP = 100
MAX_CELL_TYPES = 2

# group is the selected type's cell table, sorted by cell_id.
rng = np.random.default_rng()
group = group.iloc[rng.permutation(len(group))].reset_index(drop=True)
levels = list(range(MIN_CELLS, len(group) + 1, CELL_STEP))
if levels[-1] != len(group):
    levels.append(len(group))

subsets = {n: group.iloc[:n] for n in levels}
```

The ODC table above contains 1,245 cells. With 100-cell increments,
the condition sizes are 100, 200, …, 1,200 and all 1,245 cells.

The notebook writes `<condition>.cell_table.tsv` for every subset and
`downsampling_manifest.csv` with the original type, selected count,
cell-table path and intended raw Cooler path.


<a id="Raw-pseudo-bulk-preparation"></a>

## 2. Raw pseudo-bulk preparation


<a id="4.-Merge-each-selected-raw-cell-set"></a>

### 2.1. Merge each selected raw cell set

For a 10-kb downstream task, a subset is aggregated with:

```bash
hicluster merge-cell-raw \
  --cell_table "${BENCHMARK_ROOT}/data/lee2019/ODC_raw_10k_line/cell_table_100.txt" \
  --chrom_size_path "${BENCHMARK_ROOT}/code/resources/genome/hg38.autosomal.chrom.sizes" \
  --output_file "$MERGED_RAW_OUTPUT" \
  --chr1 1 --pos1 2 --chr2 5 --pos2 6 \
  --resolution 10000 --min_pos_dist 5000
```

Repeat for the saved conditions, including `all`. Select the resolution
required by the downstream analysis. Keep each raw pseudo-bulk as the matched
baseline for that cell set. Here, raw means non-imputed contact data after
the specified filtering and aggregation.


<a id="5.-Organize-the-Lee2019-pseudo-bulk-inputs"></a>

### 2.2. Organize the Lee2019 pseudo-bulk inputs

The Lee2019 ODC example uses a 10-kb hg38 grid. Keep the cell-number label
with each Raw and imputed matrix:

| File | Dataset and condition | Role |
|---|---|---|
| `ODC.raw.100.cool` | Lee2019 ODC, 100 cells | Raw query matrix |
| `ODC.impute.100.iEnhance_10k.cool` | Lee2019 ODC, the matched 100 cells | iEnhance query matrix |
| `ODC.raw.all.cool` | Lee2019 ODC, all cells | Common Raw map for APA |
| `hg38.autosomal.chrom.sizes` | hg38, chromosomes 1–22 | Genome coordinates |

Keep the selected cell IDs, cell-type label, cell number and resolution
with each matrix. Match these identities between the Raw and imputed
query maps, and use the all-cell Raw map as the common APA input.


<a id="Imputation-workflows"></a>

## 3. Imputation workflows


<a id="6.-Choose-the-imputation-branch"></a>

### 3.1. Choose the imputation branch

| Branch | Starting input | Output passed downstream |
|---|---|---|
| scHiCluster | Filtered single-cell contacts | Per-cell imputed Coolers and their cell table |
| Higashi | Ordered native single-cell inputs and JSON configuration | Native embeddings and chromosome-specific imputed HDF5 files |
| Pseudo-bulk methods | A raw merged Cooler for a selected cell set | An enhanced Cooler for that same set |

Use saved subset IDs when matching raw and imputed conditions. A contact-file
table, a Cooler table and a Higashi native input directory have different
formats, even when they represent the same biological cells.

The examples below show how inputs are constructed. The downstream tutorials
explain how each task consumes the relevant output.


<a id="7.-Prepare-and-run-scHiCluster-imputation"></a>

### 3.2. Prepare and run scHiCluster imputation

The single-cell implementation ([data_preparation/single_cell_imputation/impute.py](https://github.com/rawandy2001-design/benchmark/blob/main/data_preparation/single_cell_imputation/impute.py) in this folder) prepares
jobs from the filtered contact table. Its 100-kb command is:

```bash
hicluster prepare-impute \
  --cell_table "$FILTERED_CONTACT_TABLE" \
  --chrom_size_path "${BENCHMARK_ROOT}/code/resources/genome/hg38.autosomal.chrom.sizes" \
  --output_dir "$IMPUTE_OUTPUT_DIR" \
  --batch_size 1272 --pad 1 --cpu_per_job 48 \
  --chr1 1 --pos1 2 --chr2 5 --pos2 6 \
  --output_dist 500000000 --window_size 500000000 --step_size 500000000 \
  --resolution 100000
```

Then run the generated command script:

```bash
bash "${IMPUTE_OUTPUT_DIR}/snakemake_cmd.txt"
```

The imputation script is named `snakemake_cmd.txt` in this workflow. The
native Loop preparation uses its separate `snakemake_cmds.txt` entry.
The imputed Coolers and generated cell table supply downstream scHiCluster
embedding or structural callers. Preserve their association with cell IDs.

The recorded 100-kb imputed-cell table begins with:

Source: `cell_table_100K.tsv` (first 2 lines).

```text
181218_snm3Cseq_hs_21yr_BA10_UMB5577_1_UMB5577_2_A10_AD001	/home/ygc/scHiCluster_Lee2019_hg38/impute/100K/chunk0/181218_snm3Cseq_hs_21yr_BA10_UMB5577_1_UMB5577_2_A10_AD001.cool
181218_snm3Cseq_hs_21yr_BA10_UMB5577_1_UMB5577_2_A10_AD004	/home/ygc/scHiCluster_Lee2019_hg38/impute/100K/chunk0/181218_snm3Cseq_hs_21yr_BA10_UMB5577_1_UMB5577_2_A10_AD004.cool
```

The corresponding generated command file contains:

Source: `snakemake_cmd.txt` (first 2 lines).

```text
snakemake -d /home/ygc/scHiCluster_Lee2019_hg38/impute/100K/chunk0 --snakefile /home/ygc/scHiCluster_Lee2019_hg38/impute/100K/chunk0/Snakefile -j 48
snakemake -d /home/ygc/scHiCluster_Lee2019_hg38/impute/100K/chunk1 --snakefile /home/ygc/scHiCluster_Lee2019_hg38/impute/100K/chunk1/Snakefile -j 48
```


<a id="8.-Run-Higashi-preprocessing-and-imputation"></a>

### 3.3. Run Higashi preprocessing and imputation

Choose the matching JSON under
`single_cell_imputation/configs/`, then configure `data_dir`, `temp_dir`,
`genome_reference_path`, `cytoband_path` and the native input files.
The repository provides `Lee2019_higashi_100K.json`, `500K.json` and `1M.json`.
They retain `resolution_cell=500000` and `dimensions=128`.

```bash
python data_preparation/single_cell_imputation/scripts/higashi_impute.py \
  --config "${BENCHMARK_ROOT}/data/higashi/embedding_all_autosomes_example_42cells/native/Higashi/attempt01/config.json"
```

The runner performs native preprocessing, embedding training, zero-neighbor imputation and five-neighbor imputation in sequence.

Raw sparse matrices generated during preprocessing are used by native
downstream routines such as Higashi masking. Imputed chromosome HDF5 arrays
are separate outputs. The Domain/TAD tutorial identifies both kinds of input.


<a id="9.-Enhance-a-Lee2019-pseudo-bulk-matrix"></a>

### 3.4. Enhance a Lee2019 pseudo-bulk matrix

Set the raw and imputed 100-cell inputs and hg38 genome resource:

```bash
RAW_100="${BENCHMARK_ROOT}/data/lee2019/ODC_raw_10k_line/ODC.raw.100.cool"
IMPUTED_100="${BENCHMARK_ROOT}/data/lee2019/bulk_impute_10k/ODC/ODC.impute.100.iEnhance_10k.cool"
CHROM_SIZES="${BENCHMARK_ROOT}/code/resources/genome/hg38.autosomal.chrom.sizes"
```

<a id="iEnhance"></a>

**iEnhance**

```bash
cooler balance "$RAW_100"
python "$IENHANCE_SCRIPT" \
  --fn "$RAW_100" \
  --of "$IMPUTED_100" \
  --cf "$CHROM_SIZES" --res 10000
```

<a id="HiCFoundation"></a>

**HiCFoundation**

Download the checkpoints from the official
[Hugging Face model repository](https://huggingface.co/wang3702/hicfoundation_models/tree/main),
as listed in the [HiCFoundation installation instructions](https://github.com/Noble-Lab/HiCFoundation#4-download-the-trained-hicfoundation-model).
Store them in the `hicfoundation_model/` directory under the HiCFoundation installation.

```bash
cd "${BENCHMARK_ROOT}/software/HiCFoundation"
mkdir -p hicfoundation_model
cd hicfoundation_model
wget https://huggingface.co/wang3702/hicfoundation_models/resolve/main/hicfoundation_pretrain.pth.tar
wget https://huggingface.co/wang3702/hicfoundation_models/resolve/main/hicfoundation_reproducibility.pth.tar
wget https://huggingface.co/wang3702/hicfoundation_models/resolve/main/hicfoundation_loop.pth.tar
wget https://huggingface.co/wang3702/hicfoundation_models/resolve/main/hicfoundation_loop_lc.pth.tar
wget https://huggingface.co/wang3702/hicfoundation_models/resolve/main/hicfoundation_resolution.pth.tar
wget https://huggingface.co/wang3702/hicfoundation_models/resolve/main/hicfoundation_epigenomic.pth.tar
wget https://huggingface.co/wang3702/hicfoundation_models/resolve/main/hicfoundation_schic.pth.tar
cd ..
```

| Workflow | Task | Checkpoint relative to the installation |
|---|---|---|
| Pseudo-bulk imputation | 3 | `hicfoundation_model/hicfoundation_resolution.pth.tar` |
| Loop calling in this workflow | 2 | `hicfoundation_model/hicfoundation_loop.pth.tar` |

For task-3 imputation, set the script and model paths explicitly:

```bash
HICFOUNDATION_ROOT="${BENCHMARK_ROOT}/software/HiCFoundation"
HICFOUNDATION_SCRIPT="${HICFOUNDATION_ROOT}/inference.py"
HICFOUNDATION_MODEL="${HICFOUNDATION_ROOT}/hicfoundation_model/hicfoundation_resolution.pth.tar"
```

```bash
python "$HICFOUNDATION_SCRIPT" \
  --input "$RAW_100" --batch_size 4 --resolution 10000 \
  --task 3 --input_row_size 224 --input_col_size 224 --stride 100 --bound 500 \
  --model_path "$HICFOUNDATION_MODEL" \
  --output "$HICFOUNDATION_OUTPUT_DIR" --genome_id hg38
```

Rename `HiCFoundation_enhanced.cool` to the method-specific benchmark
filename and balance the final Cooler.

<a id="HiCNN2"></a>

**HiCNN2**

Process human autosomes 1–22. For each chromosome, extract submatrices,
run model 3 with ratio 16 and combine predictions. The chromosome-1 commands are:

```bash
python "${HICNN2_ROOT}/get_HiCNN2_input_fromCool.py" \
  "$RAW_100" 1 10000 ${HICNN2_WORK_DIR}/chr1.submats ${HICNN2_WORK_DIR}/chr1.index 22
python "${HICNN2_ROOT}/HiCNN2_predict.py" \
  -f1 ${HICNN2_WORK_DIR}/chr1.submats.npy -f2 ${HICNN2_WORK_DIR}/chr1.subMats_HiCNN23_16 \
  -mid 3 -m "$HICNN2_MODEL" -r 16
python "${HICNN2_ROOT}/combine_subMats.py" \
  ${HICNN2_WORK_DIR}/chr1.subMats_HiCNN23_16.npy ${HICNN2_WORK_DIR}/chr1.index.npy \
  1 10000 ${HICNN2_WORK_DIR}/predicted/chr1.predictedMat 22
```

After processing the 22 autosomes, merge the chromosome predictions:

```bash
python "${HICNN2_ROOT}/merge_chroms.py" ${HICNN2_WORK_DIR}/predicted/ ODC 100 \
  "$HICNN2_OUTPUT_COOL" \
  "$CHROM_SIZES" 10000
```

<a id="HiCPlus"></a>

**HiCPlus**

```bash
python "$HICPLUS_SCRIPT" \
  --input "$RAW_100" \
  --output "$HICPLUS_OUTPUT_COOL" \
  --chromsizes "$CHROM_SIZES" \
  --model "$HICPLUS_MODEL" --res 10000
```
